#### 07. Training and tuning

Logistic regression and random forest from `task2/code/models.py`, trained on the subsampled training table (2017–2022, dry rows weighted back up with `w`) and tuned on 2023 by RPS. Both output the 9 class probabilities; the threshold mapper turns them into P(Y > θ) for any θ. The test year 2024 is not used here.

**Setup**

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../code")
import pandas as pd
from pyspark.sql import functions as F

import config as C
import evaluate as E
import mapper as M
import models as MD

spark = C.get_spark("task2-training")

TRAIN_NAME = "train_frac0.1"
train = spark.read.parquet(str(C.ASSEMBLED / TRAIN_NAME))
val = spark.read.parquet(str(C.ASSEMBLED / "val"))
ALL = MD.feature_columns(train, ("t0", "t1", "t2"))
print(len(ALL), "features")

41 features


**Threshold mapper**

The mapper stores the rainfall values inside each wet class of the training table, so P(Y > θ) can be read off for any θ. It is part of the trained model.

In [2]:
mp = M.fit(train)
M.save(mp, C.MODELS / "mapper.json")
{k: (len(v["y"]), max(v["y"])) for k, v in mp["classes"].items()}

{1: (1, 0.2),
 2: (1, 0.4),
 3: (3, 1.0),
 4: (5, 2.0),
 5: (27, 5.0),
 6: (50, 10.0),
 7: (100, 20.0),
 8: (392, 71.8)}

**Tuning on 2023**

Both models use all 41 features. Logistic regression is tuned over the strength of the L2 penalty, the random forest over the depth of its 50 trees, and the setting with the lowest RPS on 2023 is kept.

In [3]:
def val_rps(model):
    return E.brier_rps(E.add_exceedance(model.transform(val)))[1]

lr_fits = {reg: MD.build_lr(ALL, reg_param=reg).fit(train) for reg in [1e-5, 1e-4, 1e-3]}
lr_rps = pd.Series({reg: val_rps(m) for reg, m in lr_fits.items()}, name="val RPS").rename_axis("regParam")
LR_REG = lr_rps.idxmin()
lr_rps.round(6)

regParam
0.00001    0.096356
0.00010    0.096411
0.00100    0.096729
Name: val RPS, dtype: float64

In [4]:
rf_fits = {depth: MD.build_rf(ALL, max_depth=depth).fit(train) for depth in [10, 12, 14]}
rf_rps = pd.Series({d: val_rps(m) for d, m in rf_fits.items()}, name="val RPS").rename_axis("maxDepth")
RF_DEPTH = rf_rps.idxmin()
rf_rps.round(6)

maxDepth
10    0.094972
12    0.093532
14    0.092746
Name: val RPS, dtype: float64

**Ablation by tier**

With the chosen settings, both models are also trained on Tier 0 alone and on Tiers 0 and 1, and all six are scored against the baselines on 2023, overall and split by whether the station is dry or wet now.

In [5]:
models = {("LR", "T0+T1+T2"): lr_fits[LR_REG], ("RF", "T0+T1+T2"): rf_fits[RF_DEPTH]}
for tiers in [("t0",), ("t0", "t1")]:
    feats = MD.feature_columns(train, tiers)
    name = "+".join(t.upper() for t in tiers)
    models[("LR", name)] = MD.build_lr(feats, reg_param=LR_REG).fit(train)
    models[("RF", name)] = MD.build_rf(feats, max_depth=RF_DEPTH).fit(train)

rows = []
for (kind, tiers), model in models.items():
    pred = model.transform(val).select("label", "y_mm", "w", "regime", "probability", "prediction").cache()
    rows.append({"model": f"{kind} {tiers}", "subset": "all", **E.score(pred)})
    for regime in ["dry_now", "wet_now"]:
        rows.append({"model": f"{kind} {tiers}", "subset": regime, **E.score(pred.filter(F.col("regime") == regime))})
    pred.unpersist()

baselines = pd.read_csv(C.RESULTS / "baselines_val.csv").rename(columns={"forecast": "model"})
results = pd.concat([baselines, pd.DataFrame(rows)]).set_index(["subset", "model"]).sort_index()
results.to_csv(C.RESULTS / "models_val.csv")
results.loc["all"].sort_values("RPS").round(4)

,RPS,log loss,Brier 0,Brier 1,Brier 5,Brier 20,AUC-PR 0,AUC-PR 1,AUC-PR 5,AUC-PR 20
model,,,,,,,,,,
RF T0+T1+T2,0.0927,0.1951,0.0258,0.0127,0.0058,0.0010,0.8066,0.6995,0.5062,0.1990
LR T0+T1+T2,0.0964,0.2021,0.0270,0.0132,0.0058,0.0010,0.7907,0.6784,0.4996,0.1847
RF T0+T1,0.1118,0.2385,0.0327,0.0149,0.0063,0.0010,0.7002,0.5851,0.4147,0.1711
LR T0+T1,0.1140,0.2403,0.0332,0.0154,0.0064,0.0010,0.6905,0.5680,0.3844,0.1349
persistence,0.1266,0.2640,0.0360,0.0173,0.0073,0.0011,0.6589,0.4815,0.2600,0.0542
climatology,0.1935,0.3615,0.0607,0.0246,0.0086,0.0011,0.1403,0.0621,0.0227,0.0030
RF T0,0.1946,0.3595,0.0612,0.0246,0.0086,0.0011,0.1275,0.0590,0.0244,0.0038
LR T0,0.1947,0.3585,0.0612,0.0247,0.0086,0.0011,0.1209,0.0539,0.0203,0.0033
constant,0.1977,0.3689,0.0624,0.0249,0.0086,0.0011,0.0668,0.0256,0.0087,0.0011


In [6]:
cols = ["RPS", "log loss", "Brier 0", "AUC-PR 0", "AUC-PR 5"]
pd.concat({"dry now": results.loc["dry_now"][cols], "wet now": results.loc["wet_now"][cols]}, axis=1).sort_values(("dry now", "RPS")).round(4)

dry now                                    wet now           \
                RPS log loss Brier 0 AUC-PR 0 AUC-PR 5     RPS log loss   
model                                                                     
RF T0+T1+T2  0.0538   0.1132  0.0187   0.4143   0.1630  0.6359   1.3359   
LR T0+T1+T2  0.0551   0.1153  0.0192   0.3828   0.1519  0.6709   1.4104   
LR T0+T1     0.0662   0.1468  0.0238   0.1146   0.0083  0.7799   1.5423   
RF T0+T1     0.0663   0.1494  0.0239   0.1253   0.0089  0.7444   1.4786   
persistence  0.0678   0.1623  0.0250   0.0256   0.0028  0.9422   1.6764   
RF T0        0.0700   0.1712  0.0260   0.0514   0.0093  1.9315   2.9861   
LR T0        0.0703   0.1713  0.0260   0.0473   0.0073  1.9300   2.9696   
climatology  0.0705   0.1714  0.0261   0.0538   0.0089  1.9093   3.0117   
constant     0.0706   0.1765  0.0262   0.0256   0.0028  1.9703   3.0531   

                                       
            Brier 0 AUC-PR 0 AUC-PR 5  
model                                  
RF T0+T1+T2  0.1243   0.9432   0.6247  
LR T0+T1+T2  0.1345   0.9329   0.6242  
LR T0+T1     0.1643   0.9026   0.5397  
RF T0+T1     0.1558   0.9125   0.5754  
persistence  0.1880   0.8341   0.3515  
RF T0        0.5529   0.6625   0.1177  
LR T0        0.5520   0.6610   0.1230  
climatology  0.5437   0.6635   0.1152  
constant     0.5680   0.6416   0.0911

**Calibration and feature importance**

In [7]:
best = {"LR": models[("LR", "T0+T1+T2")], "RF": models[("RF", "T0+T1+T2")]}
tables = {}
for kind, model in best.items():
    ex = E.add_exceedance(model.transform(val))
    for th in [0.0, 5.0]:
        tables[f"{kind}, θ = {th:g} mm"] = E.reliability(ex, th).set_index("bin")[["n", "mean_forecast", "observed"]]
pd.concat(tables, axis=1).round(3)

LR, θ = 0 mm                        LR, θ = 5 mm                         \
               n mean_forecast observed            n mean_forecast observed   
bin                                                                           
0      5912246.0         0.010    0.010    6601005.0         0.003    0.003   
1       246542.0         0.141    0.151      44226.0         0.142    0.153   
2       110524.0         0.245    0.279      21402.0         0.246    0.241   
3        71280.0         0.347    0.383      14077.0         0.347    0.331   
4        52859.0         0.448    0.468      10391.0         0.448    0.431   
5        44354.0         0.549    0.557       8234.0         0.548    0.512   
6        42191.0         0.650    0.653       6909.0         0.649    0.603   
7        45152.0         0.750    0.763       5802.0         0.749    0.711   
8        48703.0         0.852    0.839       4742.0         0.849    0.841   
9       145423.0         0.973    0.969       2486.0         0.934    0.937   

    RF, θ = 0 mm                        RF, θ = 5 mm                         
               n mean_forecast observed            n mean_forecast observed  
bin                                                                          
0      5922669.0         0.011    0.010    6583050.0         0.003    0.003  
1       203078.0         0.144    0.145      61614.0         0.143    0.143  
2       119289.0         0.246    0.242      28468.0         0.244    0.260  
3        76486.0         0.347    0.349      14922.0         0.346    0.362  
4        58377.0         0.449    0.449       9513.0         0.446    0.472  
5        50552.0         0.549    0.555       7563.0         0.548    0.588  
6        45692.0         0.650    0.658       5684.0         0.648    0.702  
7        44315.0         0.750    0.758       4348.0         0.748    0.816  
8        51003.0         0.853    0.870       3513.0         0.845    0.907  
9       147813.0         0.968    0.978        599.0         0.913    0.963

In [8]:
importance = pd.Series(best["RF"].stages[-1].featureImportances.toArray(), index=ALL, name="importance")
importance.sort_values(ascending=False).head(15).round(4)

t2_idw_rain_15        0.1885
t2_wet_frac_10km      0.1233
t2_idw_rain_30        0.0922
t1_max5_30            0.0663
t1_rain_15            0.0644
t1_min_since_rain     0.0605
t1_rain_30            0.0518
t2_island_wet_frac    0.0512
t1_rain_5             0.0494
t2_wet_frac_5km       0.0491
t2_dist_rain_km       0.0412
t2_idw_trend          0.0299
t2_max_rain_30        0.0225
t2_island_rain_30     0.0154
t1_wet_slots_60       0.0143
Name: importance, dtype: float64

**Saving the models**

In [9]:
for kind, model in best.items():
    model.write().overwrite().save(str(C.MODELS / kind.lower()))
sorted(p.name for p in C.MODELS.iterdir())

['lr', 'mapper.json', 'rf']